In [203]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
import os

In [204]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [205]:
class ResNet18Encoder(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, emb_dim)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        out = F.normalize(out, dim=-1)
        return out

In [206]:
class ResNet18EncoderPretrained(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  #izuzimamo poslednji sloj (klasifikaciju)
        for param in self.backbone.parameters():
            param.requires_grad = False #fiksirani svi parametri (ne menjaju se tokom treninga)
        self.fc = nn.Linear(512, emb_dim)

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        features = features.flatten(1)
        out = self.fc(features)
        out = F.normalize(out, dim=1)
        return out
    

In [207]:
class ResNet18EncoderFineTuned(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  #izuzimamo poslednji sloj (klasifikaciju)
        for param in self.backbone.parameters():
            param.requires_grad = False #fiksirani svi parametri (ne menjaju se tokom treninga)
        for param in resnet.layer4.parameters():
            param.requires_grad = True #odmrznem l4
        self.fc = nn.Linear(512, emb_dim)

    def forward(self, x):
        features = self.backbone(x)
        features = features.flatten(1)
        out = self.fc(features)
        out = F.normalize(out, dim=1)
        return out

In [208]:
# Primer upotrebe:

# model = ResNet18Encoder()
# images, filenames = next(iter(train_loader))
# print(images.shape)
# with torch.no_grad():
#     embeddings = model(images)
# print(embeddings.shape)  